# 🚗 ENSAssuRances — Analyse des Sinistres Automobiles
## Direction Technique Automobile · Rapport d'Analyse Complet

---

**Projet :** Analyse actuarielle du portefeuille sinistres automobiles  
**Compagnie :** ENSAssuRances (fictive)  
**Données :** Tables `Contrat` (~320 000 lignes) & `Sinistre` (~67 000 lignes)  
**Objectif :** Identifier les facteurs de risque, comprendre la sinistralité et produire des insights actionnables pour la gestion des risques.

---

### 📋 Plan du notebook

| # | Section | Contenu |
|---|---------|---------|
| 1 | **Setup & Chargement** | Imports, chargement des tables nettoyées |
| 2 | **Jointures** | Pivot + left_join, inner_join, contrôle qualité |
| 3 | **Analyse Univariée** | Distributions, valeurs manquantes |
| 4 | **Évolution Temporelle** | Sinistres par année, contrats par exercice |
| 5 | **Analyse par Garantie** | Base, 0km, VHR – répartition & coût |
| 6 | **Profil Conducteur** | Âge, genre, permis, antécédents |
| 7 | **Analyse Véhicule** | Segment, énergie, groupe, marque |
| 8 | **Analyse Contractuelle** | Formule, franchise, option petit rouleur |
| 9 | **Analyse Géographique** | Zones à risque par département |
| 10 | **Fréquence & Sévérité** | Taux de sinistralité, montants, délais |
| 11 | **Véhicules à Risque** | Score composite, top profils dangereux |
| 12 | **Insights & Recommandations** | Synthèse actionnables ENSAssuRances |
| 13 | **Export** | Sauvegarde des tables analytiques |

---
## 1. 🔧 Setup & Chargement des Données

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker# ── Imports ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from IPython.display import display, Markdown
import os

warnings.filterwarnings("ignore")

# ── Palette ENSAssuRances ────────────────────────────────────────────────────
PALETTE   = ["#1F3A5F", "#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B4E2B", "#5C4033"]
BG_COLOR  = "#F8F9FA"
ACCENT    = "#1F3A5F"

plt.rcParams.update({
    "figure.facecolor"  : BG_COLOR,
    "axes.facecolor"    : BG_COLOR,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.labelsize"    : 12,
    "axes.titlesize"    : 14,
    "xtick.labelsize"   : 10,
    "ytick.labelsize"   : 10,
    "font.family"       : "DejaVu Sans",
    "figure.dpi"        : 120,
})

print("✅  Imports effectués avec succès.")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   seaborn : {sns.__version__}")

✅  Imports effectués avec succès.
   pandas  : 3.0.1
   numpy   : 2.4.4
   seaborn : 0.13.2


In [ ]:
# ── Chargement des tables nettoyées ─────────────────────────────────────────
# Ces tables sont produites par :
#   - preparation_sinistre.ipynb  → Sinistre_clean.csv
#   - preparation_contrat.ipynb   → Contrat_clean.csv

df_sin = pd.read_csv(
    "data/Sinistre_clean.csv",
    parse_dates=["date_survenance", "date_declaration", "date_cloture", "date_gestion"]
)

df_ct = pd.read_csv(
    "data/Contrat_clean.csv",
    low_memory=False,
    parse_dates=["date_debut_sit", "date_fin_sit"]
)

# Extraction de l'année de survenance
df_sin["annee_survenance"] = df_sin["date_survenance"].dt.year

print("=" * 60)
print("TABLE SINISTRE — Aperçu")
print("=" * 60)
display(df_sin.head(3))
print(f"Shape : {df_sin.shape[0]:,} lignes × {df_sin.shape[1]} colonnes\n")

print("=" * 60)
print("TABLE CONTRAT — Aperçu")
print("=" * 60)
display(df_ct.head(3))
print(f"Shape : {df_ct.shape[0]:,} lignes × {df_ct.shape[1]} colonnes")

---
## 2. 🔗 Jointures des Tables

La table **Contrat** référence jusqu'à **9 identifiants sinistres** par ligne  
(`id1_sinistre_base`, `id1_sinistre_0km`, `id1_sinistre_vhr`, … `id3_sinistre_vhr`).

### Stratégie de jointure

| Étape | Opération | Objectif |
|-------|-----------|----------|
| **1** | `melt` des colonnes ID → format long | 1 ligne = 1 identifiant sinistre |
| **2** | `left join` contrat-long → sinistre | Table enrichie principale |
| **3** | `inner join` simplifié | Contrats avec sinistre Base uniquement |
| **4** | Flag `a_sinistre` sur df_ct | Calcul de fréquence de sinistralité |

In [ ]:
# ── Étape 1 : Identification des colonnes ID sinistres ────────────────────────
id_cols = [c for c in df_ct.columns if c.startswith("id") and "sinistre" in c]
print(f"Colonnes ID sinistres détectées ({len(id_cols)}) :")
print(id_cols)

In [ ]:
# ── Étape 2 : Dépivotage (wide → long) ───────────────────────────────────────
meta_cols = [c for c in df_ct.columns if c not in id_cols]

df_ct_long = df_ct.melt(
    id_vars    = meta_cols,
    value_vars = id_cols,
    var_name   = "source_id_col",
    value_name = "idx_sin"
).dropna(subset=["idx_sin"])

# Extraction du rang (id1/id2/id3) et du type de garantie (base/0km/vhr)
df_ct_long["rang_sinistre"] = df_ct_long["source_id_col"].str.extract(r"id(\d)")[0]
df_ct_long["type_sinistre"] = df_ct_long["source_id_col"].str.extract(r"sinistre_(\w+)")[0]

print(f"Table contrat long (dépivotée) : {df_ct_long.shape[0]:,} lignes × {df_ct_long.shape[1]} colonnes")
print(f"  → Chaque ligne = 1 référence sinistre dans 1 contrat")
display(df_ct_long[["id_contrat","annee_exercice","vehicule_segment",
                     "source_id_col","rang_sinistre","type_sinistre","idx_sin"]].head(5))

In [ ]:
# ── Étape 3 : LEFT JOIN principal (contrat long → sinistre) ──────────────────
df_main = pd.merge(
    df_ct_long,
    df_sin,
    on       = "idx_sin",
    how      = "left",
    suffixes = ("_ct", "_sin")
)

# Déduplication : même sinistre parfois référencé par plusieurs colonnes
n_avant = len(df_main)
df_main = df_main.drop_duplicates(subset=["idx_sin", "id_contrat"])
n_apres = len(df_main)

print("═" * 60)
print("  RÉSULTAT — TABLE ENRICHIE PRINCIPALE (left join)")
print("═" * 60)
print(f"  Lignes avant déduplication : {n_avant:>10,}")
print(f"  Lignes après déduplication : {n_apres:>10,}")
print(f"  Dimensions finales         : {df_main.shape[0]:>10,} × {df_main.shape[1]}")
print("═" * 60)

In [ ]:
# ── Étape 4 : INNER JOIN (contrats avec sinistre AssBase uniquement) ─────────
df_inner = pd.merge(
    df_ct,
    df_sin,
    left_on  = "id1_sinistre_base",
    right_on = "idx_sin",
    how      = "inner",
    suffixes = ("_ct", "_sin")
)
print(f"Inner join (contrats + sinistre AssBase) : {len(df_inner):,} lignes")

# ── Étape 5 : Flag sinistre sur df_ct ────────────────────────────────────────
id_cols_ct = [c for c in df_ct.columns if c.startswith("id") and "sinistre" in c]
df_ct["a_sinistre"] = df_ct[id_cols_ct].notna().any(axis=1).astype(int)

contrats_avec = df_ct["a_sinistre"].sum()
contrats_sans = len(df_ct) - contrats_avec
freq_glob     = contrats_avec / len(df_ct)

print(f"\nContrats AVEC sinistre  : {contrats_avec:>10,}")
print(f"Contrats SANS sinistre  : {contrats_sans:>10,}")
print(f"Taux de sinistralité    : {freq_glob*100:>9.2f} %")

In [ ]:
# ── Contrôle qualité des jointures ───────────────────────────────────────────
sin_ids_in_ct  = set(df_ct_long["idx_sin"].dropna())
sin_ids_in_sin = set(df_sin["idx_sin"].dropna())

retrouves  = sin_ids_in_ct & sin_ids_in_sin
orphelins  = sin_ids_in_sin - sin_ids_in_ct
non_decl   = sin_ids_in_ct - sin_ids_in_sin

print("╔══════════════════════════════════════════════════════╗")
print("║       CONTRÔLE QUALITÉ DES JOINTURES                ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  IDs sinistres dans Sinistre        : {len(sin_ids_in_sin):>8,}          ║")
print(f"║  IDs sinistres dans Contrat         : {len(sin_ids_in_ct):>8,}          ║")
print(f"║  IDs retrouvés dans les deux tables : {len(retrouves):>8,}          ║")
print(f"║  Sinistres orphelins (Sinistre only): {len(orphelins):>8,}          ║")
pct = (1 - len(orphelins)/max(len(sin_ids_in_sin),1))*100
print(f"║  Taux de couverture jointure        : {pct:>7.1f} %         ║")
print("╚══════════════════════════════════════════════════════╝")

---
## 3. 📊 Analyse Exploratoire Univariée

In [ ]:
# ── Résumé statistique — Table Contrat ───────────────────────────────────────
print("RÉSUMÉ STATISTIQUE — VARIABLES NUMÉRIQUES (Contrat)")
num_cols = df_ct.select_dtypes(include=[np.number]).columns.tolist()
display(df_ct[num_cols].describe().T.round(2))

In [ ]:
# ── Valeurs manquantes — Table Contrat ───────────────────────────────────────
miss_ct = (df_ct.isnull().mean() * 100).sort_values(ascending=False)
miss_ct = miss_ct[miss_ct > 0]

if len(miss_ct) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(miss_ct) * 0.45)))
    bars = ax.barh(miss_ct.index[::-1], miss_ct.values[::-1],
                   color=PALETTE[1], edgecolor="white", height=0.6)
    for bar, val in zip(bars, miss_ct.values[::-1]):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f"{val:.1f} %", va="center", fontsize=9, color=ACCENT)
    ax.axvline(5, color="red", linestyle="--", alpha=0.6, label="Seuil 5 %")
    ax.set_xlabel("% de valeurs manquantes")
    ax.set_title("Valeurs manquantes — Table Contrat",
                 fontweight="bold", color=ACCENT)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("✅  Aucune valeur manquante dans les colonnes clés.")

# Valeurs manquantes dans Sinistre
miss_sin = (df_sin.isnull().mean() * 100).sort_values(ascending=False)
miss_sin = miss_sin[miss_sin > 0]
print("\nValeurs manquantes — Table Sinistre :")
display(miss_sin.reset_index().rename(columns={"index": "variable", 0: "% manquant"}).round(2))

---
## 4. 📅 Évolution Temporelle des Sinistres

In [ ]:
# ── Sinistres par année de survenance ────────────────────────────────────────
sin_year = (
    df_sin
    .dropna(subset=["annee_survenance"])
    .astype({"annee_survenance": int})
    .query("2018 <= annee_survenance <= 2024")
    .groupby("annee_survenance")
    .agg(
        nb_sinistres     = ("idx_sin", "count"),
        montant_eval_tot = ("montant_evaluation", "sum"),
        montant_regl_tot = ("montant_regle", "sum"),
        montant_eval_moy = ("montant_evaluation", "mean"),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogramme du nombre de sinistres
ax = axes[0]
bars = ax.bar(sin_year["annee_survenance"], sin_year["nb_sinistres"],
              color=PALETTE[1], edgecolor="white", linewidth=1.2, width=0.7)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f"{bar.get_height():,.0f}", ha="center", va="bottom", fontsize=9, color=ACCENT)
ax.set_title("Nombre total de sinistres par année", fontweight="bold", color=ACCENT)
ax.set_xlabel("Année de survenance")
ax.set_ylabel("Nombre de sinistres")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Montants évalués vs réglés
ax2 = axes[1]
w = 0.35
x = np.arange(len(sin_year))
ax2.bar(x - w/2, sin_year["montant_eval_tot"]/1e6, w, label="Évalué", color=PALETTE[0])
ax2.bar(x + w/2, sin_year["montant_regl_tot"]/1e6, w, label="Réglé",  color=PALETTE[3])
ax2.set_xticks(x)
ax2.set_xticklabels(sin_year["annee_survenance"])
ax2.set_title("Charge évaluée vs réglée (M€) par année", fontweight="bold", color=ACCENT)
ax2.set_xlabel("Année")
ax2.set_ylabel("Montant (M€)")
ax2.legend()
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f} M€"))

plt.suptitle("Évolution annuelle de la sinistralité — ENSAssuRances",
             fontsize=14, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

display(sin_year.set_index("annee_survenance")
        .style.format({
            "nb_sinistres": "{:,.0f}",
            "montant_eval_tot": "{:,.0f} €",
            "montant_regl_tot": "{:,.0f} €",
            "montant_eval_moy": "{:.2f} €"
        }).background_gradient(cmap="Blues", subset=["nb_sinistres"])
        .set_caption("Récapitulatif annuel des sinistres"))

In [ ]:
# ── Distribution des contrats par année d'exercice ───────────────────────────
ct_year = df_ct.groupby("annee_exercice").size().reset_index(name="nb_contrats")

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ct_year["annee_exercice"], ct_year["nb_contrats"],
       color=PALETTE[0], edgecolor="white", width=0.7)
for _, row in ct_year.iterrows():
    ax.text(row["annee_exercice"], row["nb_contrats"] + 500,
            f"{row['nb_contrats']:,}", ha="center", fontsize=9, color=ACCENT)
ax.set_title("Distribution des contrats par année d'exercice",
             fontweight="bold", color=ACCENT)
ax.set_xlabel("Année d'exercice")
ax.set_ylabel("Nombre de contrats")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

---
## 5. 🛡️ Analyse par Type de Garantie

In [ ]:
# ── Répartition et coût par garantie ────────────────────────────────────────
label_map = {
    "AssBase": "Assistance de Base",
    "Ass0km": "Assistance 0 km",
    "AssVHR": "Véhicule de Remplacement",
    "Assistance de base": "Assistance de Base",
    "Assistance 0 km": "Assistance 0 km",
    "Véhicule de remplacement": "Véhicule de Remplacement"
}

df_sin["garantie_label"] = df_sin["type_garantie_sinistree"].map(label_map).fillna(df_sin["type_garantie_sinistree"])
gar = df_sin["garantie_label"].value_counts()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Camembert
wedges, texts, autotexts = axes[0].pie(
    gar.values, labels=gar.index, autopct="%1.1f%%", startangle=140,
    colors=PALETTE[:3], wedgeprops={"edgecolor":"white","linewidth":1.5}
)
for at in autotexts:
    at.set_fontsize(11)
axes[0].set_title("Répartition des sinistres par garantie", fontweight="bold", color=ACCENT)

# Coût moyen par garantie
cout_gar = df_sin.groupby("garantie_label")["montant_evaluation"].mean().sort_values()
axes[1].barh(cout_gar.index, cout_gar.values, color=PALETTE[:3], edgecolor="white")
for i, v in enumerate(cout_gar.values):
    axes[1].text(v + 2, i, f"{v:.0f} €", va="center", fontsize=10)
axes[1].set_xlabel("Montant moyen évalué (€)")
axes[1].set_title("Coût moyen par garantie", fontweight="bold", color=ACCENT)

# Taux de règlement
taux = df_sin.groupby("garantie_label").apply(lambda x: (x["montant_regle"]>0).mean()*100)
colors_tx = [PALETTE[4] if v < 50 else PALETTE[2] if v < 80 else PALETTE[1] for v in taux.values]
axes[2].bar(taux.index, taux.values, color=colors_tx, edgecolor="white")
axes[2].axhline(100, color="grey", linestyle="--", alpha=0.4, label="100 %")
axes[2].set_ylabel("% sinistres avec règlement > 0")
axes[2].set_title("Taux de règlement par garantie", fontweight="bold", color=ACCENT)
axes[2].set_ylim(0, 110)
for i, v in enumerate(taux.values):
    axes[2].text(i, v + 1.5, f"{v:.1f} %", ha="center", fontsize=10)

plt.suptitle("Analyse par type de garantie sinistrée — ENSAssuRances",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

# Table résumé détaillée
gar_detail = df_sin.groupby("garantie_label").agg(
    nb_sinistres     = ("idx_sin", "count"),
    montant_moy_eval = ("montant_evaluation", "mean"),
    montant_moy_regl = ("montant_regle", "mean"),
    taux_reglement   = ("montant_regle", lambda x: (x > 0).mean() * 100),
    montant_total    = ("montant_evaluation", "sum")
).round(2)
display(gar_detail.style.format("{:,.2f}")
        .background_gradient(cmap="Blues", subset=["nb_sinistres"])
        .set_caption("Détail par type de garantie"))

---
## 6. 👤 Profil du Conducteur & Sinistralité

In [ ]:
# ── Sinistralité selon l'âge du conducteur ───────────────────────────────────
bins_age = [18, 25, 35, 45, 55, 65, 100]
lbls_age = ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"]

df_main["tranche_age"] = pd.cut(df_main["age_conducteur"], bins=bins_age, labels=lbls_age, right=False)
df_ct["tranche_age"]   = pd.cut(df_ct["age_conducteur"],   bins=bins_age, labels=lbls_age, right=False)

# Sinistres par tranche d'âge
sin_age = df_main.groupby("tranche_age", observed=True).agg(
    nb_sinistres = ("idx_sin", "count"),
    moy_eval     = ("montant_evaluation", "mean"),
    nb_contrats  = ("id_contrat", "nunique")
).reset_index()
sin_age["freq_sin"] = sin_age["nb_sinistres"] / sin_age["nb_contrats"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Nombre sinistres
colors_age = sns.color_palette("Blues_d", len(sin_age))
axes[0].bar(sin_age["tranche_age"].astype(str), sin_age["nb_sinistres"],
            color=colors_age, edgecolor="white")
axes[0].set_title("Nombre de sinistres par tranche d'âge", fontweight="bold", color=ACCENT)
axes[0].set_xlabel("Tranche d'âge du conducteur")
axes[0].set_ylabel("Nombre de sinistres")
for i, v in enumerate(sin_age["nb_sinistres"]):
    axes[0].text(i, v + 20, f"{v:,}", ha="center", fontsize=8)

# Fréquence sinistralité
axes[1].plot(sin_age["tranche_age"].astype(str), sin_age["freq_sin"],
             marker="o", color=PALETTE[2], linewidth=2.5, markersize=9)
axes[1].fill_between(range(len(sin_age)), sin_age["freq_sin"],
                     alpha=0.2, color=PALETTE[2])
axes[1].set_xticks(range(len(sin_age)))
axes[1].set_xticklabels(sin_age["tranche_age"].astype(str))
axes[1].set_title("Fréquence de sinistralité", fontweight="bold", color=ACCENT)
axes[1].set_ylabel("Sinistres / Contrat")
axes[1].set_xlabel("Tranche d'âge")

# Coût moyen
axes[2].bar(sin_age["tranche_age"].astype(str), sin_age["moy_eval"],
            color=sns.color_palette("Oranges_d", len(sin_age)), edgecolor="white")
axes[2].set_title("Coût moyen d'évaluation par tranche d'âge", fontweight="bold", color=ACCENT)
axes[2].set_ylabel("Montant moyen évalué (€)")

plt.suptitle("Impact de l'âge du conducteur sur la sinistralité",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Genre du conducteur ───────────────────────────────────────────────────────
sin_genre = df_main.groupby("genre_conducteur").agg(
    nb_sinistres = ("idx_sin", "count"),
    moy_eval     = ("montant_evaluation", "mean")
).reset_index()

ct_genre = df_ct["genre_conducteur"].value_counts().reset_index()
ct_genre.columns = ["genre_conducteur", "nb_contrats"]

sin_genre = sin_genre.merge(ct_genre, on="genre_conducteur")
sin_genre["pct_sinistres"] = sin_genre["nb_sinistres"] / sin_genre["nb_sinistres"].sum() * 100
sin_genre["freq_sin"]      = sin_genre["nb_sinistres"] / sin_genre["nb_contrats"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].pie(sin_genre["nb_contrats"], labels=sin_genre["genre_conducteur"],
            autopct="%1.1f%%", colors=PALETTE[:2],
            wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[0].set_title("Répartition des contrats par genre", fontweight="bold", color=ACCENT)

axes[1].bar(sin_genre["genre_conducteur"], sin_genre["nb_sinistres"],
            color=PALETTE[:2], edgecolor="white")
for i, (v, p) in enumerate(zip(sin_genre["nb_sinistres"], sin_genre["pct_sinistres"])):
    axes[1].text(i, v + 30, f"{v:,}\n({p:.1f}%)", ha="center", fontsize=10, color=ACCENT)
axes[1].set_title("Nombre et % de sinistres par genre", fontweight="bold", color=ACCENT)
axes[1].set_ylabel("Nombre de sinistres")

axes[2].bar(sin_genre["genre_conducteur"], sin_genre["freq_sin"],
            color=PALETTE[:2], edgecolor="white")
for i, v in enumerate(sin_genre["freq_sin"]):
    axes[2].text(i, v + 0.001, f"{v:.4f}", ha="center", fontsize=10)
axes[2].set_title("Fréquence de sinistralité par genre", fontweight="bold", color=ACCENT)
axes[2].set_ylabel("Sinistres / Contrat")

plt.suptitle("Sinistralité selon le genre du conducteur principal",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

display(sin_genre.set_index("genre_conducteur")
        .style.format({"nb_sinistres":"{:,}","moy_eval":"{:.2f} €",
                       "nb_contrats":"{:,}","pct_sinistres":"{:.1f} %","freq_sin":"{:.4f}"})
        .background_gradient(cmap="Blues", subset=["freq_sin"]))

In [ ]:
# ── Sinistres selon les sinistres antérieurs ─────────────────────────────────
sin_ant = (
    df_main
    .dropna(subset=["nb_sinistres_ant"])
    .groupby("nb_sinistres_ant")
    .agg(nb_sinistres=("idx_sin","count"), montant_moy=("montant_evaluation","mean"))
    .reset_index()
    .astype({"nb_sinistres_ant": int})
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
pal_ant = [PALETTE[1], PALETTE[3], PALETTE[4]]

axes[0].bar(sin_ant["nb_sinistres_ant"].astype(str), sin_ant["nb_sinistres"],
            color=pal_ant[:len(sin_ant)], edgecolor="white")
for i, v in enumerate(sin_ant["nb_sinistres"]):
    axes[0].text(i, v + 30, f"{v:,}", ha="center", fontsize=10)
axes[0].set_title("Nombre de sinistres selon les antécédents", fontweight="bold", color=ACCENT)
axes[0].set_xlabel("Nb sinistres antérieurs")
axes[0].set_ylabel("Nb sinistres actuels")

axes[1].bar(sin_ant["nb_sinistres_ant"].astype(str), sin_ant["montant_moy"],
            color=pal_ant[:len(sin_ant)], edgecolor="white")
for i, v in enumerate(sin_ant["montant_moy"]):
    axes[1].text(i, v + 1, f"{v:.0f} €", ha="center", fontsize=10)
axes[1].set_title("Coût moyen selon les antécédents", fontweight="bold", color=ACCENT)
axes[1].set_xlabel("Nb sinistres antérieurs")
axes[1].set_ylabel("Montant moyen évalué (€)")

plt.suptitle("Effet des antécédents sinistres sur la sinistralité actuelle",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. 🚙 Analyse des Caractéristiques du Véhicule

In [ ]:
# ── Répartition par segment commercial ──────────────────────────────────────
seg_ct  = df_ct["vehicule_segment"].value_counts()
sin_seg = (df_main.groupby("vehicule_segment")["idx_sin"].count()
           .sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_seg = PALETTE[:len(seg_ct)]
axes[0].bar(seg_ct.index, seg_ct.values, color=colors_seg, edgecolor="white")
for i, v in enumerate(seg_ct.values):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontsize=9, color=ACCENT)
axes[0].set_title("Contrats par segment commercial", fontweight="bold", color=ACCENT)
axes[0].set_ylabel("Nombre de contrats")

axes[1].barh(sin_seg.index[::-1], sin_seg.values[::-1], color=PALETTE[1], edgecolor="white")
for i, v in enumerate(sin_seg.values[::-1]):
    axes[1].text(v + 30, i, f"{v:,}", va="center", fontsize=9)
axes[1].set_title("Sinistres par segment commercial", fontweight="bold", color=ACCENT)
axes[1].set_xlabel("Nombre de sinistres")

plt.suptitle("Analyse par segment commercial — ENSAssuRances",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Alimentation (énergie) ───────────────────────────────────────────────────
ener_ct = df_ct["vehicule_energie"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(ener_ct.values, labels=ener_ct.index, autopct="%1.1f%%",
            colors=PALETTE[:len(ener_ct)],
            explode=[0.03]*len(ener_ct),
            wedgeprops={"edgecolor":"white","linewidth":1.5})
axes[0].set_title("Répartition des véhicules selon l'alimentation",
                  fontweight="bold", color=ACCENT)

sin_ener = df_main.groupby("vehicule_energie").agg(
    nb_sin   = ("idx_sin", "count"),
    moy_eval = ("montant_evaluation", "mean")
).reset_index()
x = np.arange(len(sin_ener))
w = 0.38

ax2 = axes[1]
ax2b = ax2.twinx()
b1 = ax2.bar(x - w/2, sin_ener["nb_sin"],   w, label="Nb sinistres", color=PALETTE[0])
b2 = ax2b.bar(x + w/2, sin_ener["moy_eval"], w, label="Coût moyen",   color=PALETTE[3])
ax2.set_xticks(x)
ax2.set_xticklabels(sin_ener["vehicule_energie"])
ax2.set_ylabel("Nombre de sinistres",         color=PALETTE[0])
ax2b.set_ylabel("Coût moyen évalué (€)",      color=PALETTE[3])
ax2.set_title("Sinistres et coût par alimentation", fontweight="bold", color=ACCENT)
handles = [mpatches.Patch(color=PALETTE[0], label="Nb sinistres"),
           mpatches.Patch(color=PALETTE[3], label="Coût moyen (€)")]
ax2.legend(handles=handles, loc="upper right")

plt.suptitle("Sinistralité selon l'alimentation du véhicule",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Groupe du véhicule ───────────────────────────────────────────────────────
grp_ct = df_ct["groupe_vehicule"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution des contrats
axes[0].bar(grp_ct.index.astype(str), grp_ct.values,
            color=sns.color_palette("Blues_d", len(grp_ct)), edgecolor="white")
axes[0].set_title("Distribution des véhicules selon leur groupe",
                  fontweight="bold", color=ACCENT)
axes[0].set_xlabel("Groupe du véhicule")
axes[0].set_ylabel("Nombre de contrats")
axes[0].tick_params(axis="x", rotation=45)

# Coût moyen par groupe
sin_grp = (df_main.groupby("groupe_vehicule")
           .agg(nb_sin=("idx_sin","count"), moy_eval=("montant_evaluation","mean"))
           .reset_index().sort_values("moy_eval", ascending=False))
axes[1].bar(sin_grp["groupe_vehicule"].astype(str), sin_grp["moy_eval"],
            color=sns.color_palette("Reds_d", len(sin_grp)), edgecolor="white")
axes[1].set_title("Coût moyen d'évaluation par groupe véhicule",
                  fontweight="bold", color=ACCENT)
axes[1].set_xlabel("Groupe du véhicule")
axes[1].set_ylabel("Montant moyen évalué (€)")
axes[1].tick_params(axis="x", rotation=45)

plt.suptitle("Analyse par groupe de véhicule — ENSAssuRances",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Marque du véhicule ───────────────────────────────────────────────────────
marque_ct  = df_ct["vehicule_marque"].value_counts()
marque_sin = df_main.groupby("vehicule_marque")["idx_sin"].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(marque_ct.index[::-1], marque_ct.values[::-1],
             color=PALETTE[1], edgecolor="white")
for i, v in enumerate(marque_ct.values[::-1]):
    axes[0].text(v + 100, i, f"{v:,}", va="center", fontsize=8)
axes[0].set_title("Contrats par marque du véhicule", fontweight="bold", color=ACCENT)
axes[0].set_xlabel("Nombre de contrats")

axes[1].barh(marque_sin.index[::-1], marque_sin.values[::-1],
             color=PALETTE[4], edgecolor="white")
for i, v in enumerate(marque_sin.values[::-1]):
    axes[1].text(v + 10, i, f"{v:,}", va="center", fontsize=8)
axes[1].set_title("Sinistres par marque du véhicule", fontweight="bold", color=ACCENT)
axes[1].set_xlabel("Nombre de sinistres")

plt.suptitle("Sinistralité par marque — ENSAssuRances",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. 📄 Analyse des Caractéristiques Contractuelles

In [ ]:
# ── Formule souscrite ────────────────────────────────────────────────────────
ordre    = ["Mini", "Med1", "Med2", "Maxi"]
col_frm  = "formule_contrat"
frm_ct   = df_ct[col_frm].value_counts()
frm_sin  = df_main.groupby(col_frm)["idx_sin"].count()

ct_vals  = [frm_ct.get(f, 0)  for f in ordre]
sin_vals = [frm_sin.get(f, 0) for f in ordre]
colors_frm = [PALETTE[1], PALETTE[0], PALETTE[3], PALETTE[4]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(ordre, ct_vals, color=colors_frm, edgecolor="white")
for i, v in enumerate(ct_vals):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontsize=9)
axes[0].set_title("Contrats par formule souscrite", fontweight="bold", color=ACCENT)
axes[0].set_ylabel("Nombre de contrats")

axes[1].bar(ordre, sin_vals, color=colors_frm, edgecolor="white")
for i, v in enumerate(sin_vals):
    axes[1].text(i, v + 30, f"{v:,}", ha="center", fontsize=9)
axes[1].set_title("Sinistres par formule souscrite", fontweight="bold", color=ACCENT)
axes[1].set_ylabel("Nombre de sinistres")

plt.suptitle("Analyse par formule contractuelle",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Option Petit Rouleur ─────────────────────────────────────────────────────
col_pr = "option_km" if "option_km" in df_ct.columns else "option_petit_rouleur"
pr_ct  = df_ct[col_pr].value_counts()

col_pr_m = "option_km" if "option_km" in df_main.columns else "option_petit_rouleur"
pr_sin = df_main.groupby(col_pr_m)["idx_sin"].count()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(pr_ct.index, pr_ct.values, color=[PALETTE[1], PALETTE[4]], edgecolor="white")
for i, v in enumerate(pr_ct.values):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontsize=10, color=ACCENT)
axes[0].set_title("Contrats avec / sans option Petit Rouleur", fontweight="bold", color=ACCENT)
axes[0].set_ylabel("Nombre de contrats")

axes[1].bar(pr_sin.index, pr_sin.values, color=[PALETTE[1], PALETTE[4]], edgecolor="white")
for i, v in enumerate(pr_sin.values):
    axes[1].text(i, v + 20, f"{v:,}", ha="center", fontsize=10, color=ACCENT)
axes[1].set_title("Sinistres selon l'option Petit Rouleur", fontweight="bold", color=ACCENT)
axes[1].set_ylabel("Nombre de sinistres")

plt.suptitle("Impact de l'option Petit Rouleur sur la sinistralité",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Franchise ────────────────────────────────────────────────────────────────
fr_ct  = df_ct["franchise"].value_counts().sort_index()
fr_sin = df_main.groupby("franchise")["idx_sin"].count().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(fr_ct.index.astype(str), fr_ct.values,
            color=PALETTE[:len(fr_ct)], edgecolor="white")
for i, v in enumerate(fr_ct.values):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontsize=9)
axes[0].set_title("Contrats par montant de franchise", fontweight="bold", color=ACCENT)
axes[0].set_xlabel("Franchise (€)")
axes[0].set_ylabel("Nombre de contrats")

axes[1].bar(fr_sin.index.astype(str), fr_sin.values,
            color=PALETTE[:len(fr_sin)], edgecolor="white")
for i, v in enumerate(fr_sin.values):
    axes[1].text(i, v + 30, f"{v:,}", ha="center", fontsize=9)
axes[1].set_title("Sinistres par montant de franchise", fontweight="bold", color=ACCENT)
axes[1].set_xlabel("Franchise (€)")
axes[1].set_ylabel("Nombre de sinistres")

plt.tight_layout()
plt.show()

---
## 9. 🗺️ Analyse Géographique — Zones à Risque

In [ ]:
# ── Extraction du département ────────────────────────────────────────────────
df_ct["departement"]   = df_ct["code_insee"].astype(str).str.zfill(5).str[:2]
df_main["departement"] = df_main["code_insee"].astype(str).str.zfill(5).str[:2]

depts_valides = [str(i).zfill(2) for i in range(1, 96)] + ["2A", "2B"]

df_ct_dep   = df_ct[df_ct["departement"].isin(depts_valides)]
df_main_dep = df_main[df_main["departement"].isin(depts_valides)]

dep_sin = df_main_dep.groupby("departement").agg(
    nb_sinistres  = ("idx_sin", "count"),
    montant_total = ("montant_evaluation", "sum"),
    montant_moy   = ("montant_evaluation", "mean")
).reset_index().sort_values("nb_sinistres", ascending=False)

dep_ct = df_ct_dep.groupby("departement").size().reset_index(name="nb_contrats")
dep_sin = dep_sin.merge(dep_ct, on="departement", how="left")
dep_sin["freq_sinistralite"] = dep_sin["nb_sinistres"] / dep_sin["nb_contrats"]
dep_sin["charge_par_contrat"] = dep_sin["montant_total"] / dep_sin["nb_contrats"]

print("Top 15 départements — Nombre de sinistres")
display(dep_sin.head(15).style
        .format({
            "nb_sinistres": "{:,}",
            "montant_total": "{:,.0f} €",
            "montant_moy": "{:.2f} €",
            "nb_contrats": "{:,}",
            "freq_sinistralite": "{:.4f}",
            "charge_par_contrat": "{:.2f} €"
        }).background_gradient(cmap="Reds", subset=["freq_sinistralite"])
        .bar(subset=["nb_sinistres"], color=PALETTE[1]))

In [ ]:
# ── Top 20 départements — Visualisation ──────────────────────────────────────
top20_nb   = dep_sin.head(20)
top20_freq = dep_sin.sort_values("freq_sinistralite", ascending=False).head(20)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Volume de sinistres
pal_red = sns.color_palette("Reds_d", 20)
axes[0].barh(top20_nb["departement"][::-1], top20_nb["nb_sinistres"][::-1],
             color=pal_red, edgecolor="white")
for i, (v, dep) in enumerate(zip(top20_nb["nb_sinistres"][::-1],
                                   top20_nb["departement"][::-1])):
    axes[0].text(v + 10, i, f"{v:,}", va="center", fontsize=8)
axes[0].set_title("Top 20 — Nombre de sinistres par département",
                  fontweight="bold", color=ACCENT)
axes[0].set_xlabel("Nombre de sinistres")

# Fréquence
pal_org = sns.color_palette("OrRd_d", 20)
axes[1].barh(top20_freq["departement"][::-1], top20_freq["freq_sinistralite"][::-1],
             color=pal_org, edgecolor="white")
for i, v in enumerate(top20_freq["freq_sinistralite"][::-1]):
    axes[1].text(v + 0.0005, i, f"{v:.4f}", va="center", fontsize=8)
axes[1].set_title("Top 20 — Fréquence de sinistralité par département",
                  fontweight="bold", color=ACCENT)
axes[1].set_xlabel("Sinistres / Contrat")

plt.suptitle("Cartographie du risque par département — ENSAssuRances",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. 📈 Fréquence & Sévérité

In [ ]:
# ── KPIs globaux ─────────────────────────────────────────────────────────────
total_contrats   = len(df_ct)
total_sinistres  = len(df_sin)
contrats_sin     = df_ct["a_sinistre"].sum()
freq_glob        = contrats_sin / total_contrats
sev_glob         = df_sin["montant_evaluation"].mean()
sev_mediane      = df_sin["montant_evaluation"].median()
charge_tot       = df_sin["montant_evaluation"].sum()
charge_reglee    = df_sin["montant_regle"].sum()
taux_reglement   = (df_sin["montant_regle"] > 0).mean() * 100

print("╔══════════════════════════════════════════════════════════╗")
print("║          KPIs SINISTRALITÉ — ENSAssuRances              ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Total contrats portefeuille     : {total_contrats:>12,}          ║")
print(f"║  Contrats avec sinistre          : {contrats_sin:>12,}          ║")
print(f"║  Total sinistres déclarés        : {total_sinistres:>12,}          ║")
print(f"║  Fréquence de sinistralité       : {freq_glob*100:>11.2f} %         ║")
print(f"║  Sévérité moyenne (€)           : {sev_glob:>12.2f}          ║")
print(f"║  Sévérité médiane (€)           : {sev_mediane:>12.2f}          ║")
print(f"║  Charge totale évaluée (€)      : {charge_tot:>12,.0f}          ║")
print(f"║  Charge totale réglée (€)       : {charge_reglee:>12,.0f}          ║")
print(f"║  Taux de règlement              : {taux_reglement:>11.1f} %         ║")
print("╚══════════════════════════════════════════════════════════╝")

In [ ]:
# ── Distribution des montants d'évaluation ──────────────────────────────────
eval_data = df_sin["montant_evaluation"].dropna()
eval_clean = eval_data[eval_data < eval_data.quantile(0.99)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogramme
axes[0].hist(eval_clean, bins=50, color=PALETTE[1], edgecolor="white", alpha=0.85)
axes[0].axvline(eval_data.mean(),   color="red",    linestyle="--",
                label=f"Moyenne : {eval_data.mean():.0f} €")
axes[0].axvline(eval_data.median(), color="orange", linestyle="--",
                label=f"Médiane : {eval_data.median():.0f} €")
axes[0].set_title("Distribution des montants évalués", fontweight="bold", color=ACCENT)
axes[0].set_xlabel("Montant (€)")
axes[0].legend()

# Boxplot par garantie
df_sin_lab = df_sin.copy()
df_sin_lab["garantie_label"] = df_sin_lab["type_garantie_sinistree"].map(label_map).fillna(
    df_sin_lab["type_garantie_sinistree"])
gars = df_sin_lab["garantie_label"].unique()
bp_data   = [df_sin_lab[df_sin_lab["garantie_label"]==g]["montant_evaluation"].dropna() for g in gars]
bp = axes[1].boxplot(bp_data, labels=gars, patch_artist=True,
                     boxprops=dict(facecolor=PALETTE[1], alpha=0.7),
                     medianprops=dict(color="white", linewidth=2))
axes[1].set_title("Montants par type de garantie", fontweight="bold", color=ACCENT)
axes[1].set_ylabel("Montant (€)")
axes[1].tick_params(axis="x", rotation=15)

# Scatter réglé vs évalué
sample = df_sin.sample(min(3000, len(df_sin)), random_state=42)
axes[2].scatter(sample["montant_evaluation"], sample["montant_regle"],
                alpha=0.3, color=PALETTE[2], s=15)
mx = max(sample["montant_evaluation"].max(), sample["montant_regle"].max())
axes[2].plot([0, mx], [0, mx], "r--", linewidth=1.5, label="Parité")
axes[2].set_title("Montant réglé vs évalué", fontweight="bold", color=ACCENT)
axes[2].set_xlabel("Évalué (€)")
axes[2].set_ylabel("Réglé (€)")
axes[2].legend()

plt.suptitle("Distribution & Sévérité des sinistres — ENSAssuRances",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Délais de gestion ────────────────────────────────────────────────────────
df_sin["delai_declaration"] = (df_sin["date_declaration"] - df_sin["date_survenance"]).dt.days
df_sin["delai_cloture"]     = (df_sin["date_cloture"]     - df_sin["date_declaration"]).dt.days

del_decl = df_sin["delai_declaration"].dropna()
del_clo  = df_sin["delai_cloture"].dropna()

del_decl_c = del_decl[(del_decl >= 0) & (del_decl < 365)]
del_clo_c  = del_clo[(del_clo >= 0)  & (del_clo < 1500)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(del_decl_c, bins=30, color=PALETTE[0], edgecolor="white", alpha=0.85)
axes[0].axvline(del_decl_c.mean(), color="red", linestyle="--",
                label=f"Moy. : {del_decl_c.mean():.1f} j")
axes[0].set_title("Délai de déclaration (survenance → déclaration)",
                  fontweight="bold", color=ACCENT)
axes[0].set_xlabel("Jours")
axes[0].legend()

axes[1].hist(del_clo_c, bins=40, color=PALETTE[3], edgecolor="white", alpha=0.85)
axes[1].axvline(del_clo_c.mean(), color="red", linestyle="--",
                label=f"Moy. : {del_clo_c.mean():.1f} j")
axes[1].set_title("Délai de clôture (déclaration → clôture)",
                  fontweight="bold", color=ACCENT)
axes[1].set_xlabel("Jours")
axes[1].legend()

plt.suptitle("Analyse des délais de gestion des sinistres",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

print(f"Délai moyen de déclaration  : {del_decl_c.mean():.1f} j  (médiane : {del_decl_c.median():.0f} j)")
print(f"Délai moyen de clôture      : {del_clo_c.mean():.1f} j  (médiane : {del_clo_c.median():.0f} j)")

In [ ]:
# ── Matrice de corrélation — Variables numériques ────────────────────────────
num_candidates = ["age_conducteur", "anciennete_permis", "age_vehicule",
                  "vehicule_poids", "puissance_din", "valeur_vehicule",
                  "exposition", "nb_sinistres_ant",
                  "montant_evaluation", "montant_regle"]

available = [c for c in num_candidates if c in df_main.columns]
corr = df_main[available].dropna().corr()

fig, ax = plt.subplots(figsize=(12, 9))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1, vmax=1, center=0,
            annot=True, fmt=".2f", linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Matrice de corrélation — Variables numériques clés",
             fontsize=14, fontweight="bold", color=ACCENT, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution des cotisations par garantie ────────────────────────────────
cot_items = [("cotisation_base","Assistance de Base"),
             ("cotisation_0km", "Assistance 0 km"),
             ("cotisation_vhr", "Assistance VHR")]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (col, label) in enumerate(cot_items):
    data = df_ct[col].dropna()
    data_pos = data[data > 0]
    if len(data_pos) > 0:
        axes[i].hist(data_pos, bins=40, color=PALETTE[i], edgecolor="white", alpha=0.85)
        axes[i].axvline(data_pos.mean(), color="red", linestyle="--",
                        label=f"Moy. : {data_pos.mean():.2f} €")
    axes[i].set_title(f"Cotisation — {label}", fontweight="bold", color=ACCENT)
    axes[i].set_xlabel("Cotisation (€)")
    axes[i].legend(fontsize=9)

plt.suptitle("Distribution des cotisations par garantie",
             fontsize=13, fontweight="bold", color=ACCENT, y=1.02)
plt.tight_layout()
plt.show()

cot_stats = pd.DataFrame({
    "Garantie": [l for _, l in cot_items],
    "Nb contrats avec cotisation > 0": [
        (df_ct[c] > 0).sum() for c, _ in cot_items
    ],
    "Cotisation moy (€)": [
        df_ct[c][df_ct[c]>0].mean() for c, _ in cot_items
    ],
    "Cotisation médiane (€)": [
        df_ct[c][df_ct[c]>0].median() for c, _ in cot_items
    ],
    "Cotisation max (€)": [
        df_ct[c].max() for c, _ in cot_items
    ],
})
display(cot_stats.style.format({
    "Nb contrats avec cotisation > 0": "{:,}",
    "Cotisation moy (€)": "{:.2f}",
    "Cotisation médiane (€)": "{:.2f}",
    "Cotisation max (€)": "{:.2f}"
}))

---
## 11. ⚠️ Identification des Véhicules & Profils à Risque Élevé

In [ ]:
# ── Score de risque composite ────────────────────────────────────────────────
risk = df_main.groupby(["vehicule_marque","vehicule_segment","vehicule_energie"]).agg(
    nb_sinistres   = ("idx_sin", "count"),
    nb_contrats    = ("id_contrat", "nunique"),
    moy_eval       = ("montant_evaluation", "mean"),
    total_eval     = ("montant_evaluation", "sum"),
    age_moy_cond   = ("age_conducteur", "mean")
).reset_index()

risk["freq_sinistralite"]  = risk["nb_sinistres"]  / risk["nb_contrats"].replace(0, np.nan)
risk["charge_par_contrat"] = risk["total_eval"]    / risk["nb_contrats"].replace(0, np.nan)

# Normalisation et pondération
for col, w in [("freq_sinistralite", 0.40), ("moy_eval", 0.35), ("nb_sinistres", 0.25)]:
    mx = risk[col].max()
    risk[f"s_{col}"] = (risk[col] / mx * 10 * w) if mx > 0 else 0.0

risk["score_risque"] = risk[["s_freq_sinistralite","s_moy_eval","s_nb_sinistres"]].sum(axis=1)
risk = risk.sort_values("score_risque", ascending=False).reset_index(drop=True)
risk["classe_risque"] = pd.cut(risk["score_risque"],
                               bins=[0, 2, 4, 10],
                               labels=["Faible", "Moyen", "Elevé"])

print("TOP 20 — Profils véhicules à risque élevé")
display(risk[["vehicule_marque","vehicule_segment","vehicule_energie",
              "nb_sinistres","freq_sinistralite","moy_eval",
              "score_risque","classe_risque"]].head(20)
        .style.format({
            "nb_sinistres": "{:,}",
            "freq_sinistralite": "{:.4f}",
            "moy_eval": "{:.2f} €",
            "score_risque": "{:.2f}"
        }).background_gradient(cmap="RdYlGn_r", subset=["score_risque"]))

In [ ]:
# ── Visualisation top 15 véhicules à risque ──────────────────────────────────
top15 = risk.head(15).copy()
top15["label"] = top15["vehicule_marque"] + " – " + top15["vehicule_segment"]

color_map = {"Elevé": PALETTE[4], "Moyen": PALETTE[3], "Faible": PALETTE[1]}
bar_colors = [color_map.get(str(r), PALETTE[1]) for r in top15["classe_risque"]]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top15["label"][::-1], top15["score_risque"][::-1],
               color=bar_colors[::-1], edgecolor="white", height=0.65)
for bar, v in zip(bars, top15["score_risque"][::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f"{v:.2f}", va="center", fontsize=9, color=ACCENT)

ax.set_xlabel("Score de risque composite (0–10)")
ax.set_title("Top 15 profils véhicules à risque\n(score = fréquence × 40% + sévérité × 35% + volume × 25%)",
             fontweight="bold", color=ACCENT, fontsize=12)

legend_p = [mpatches.Patch(color=PALETTE[4], label="Risque élevé"),
            mpatches.Patch(color=PALETTE[3], label="Risque moyen"),
            mpatches.Patch(color=PALETTE[1], label="Risque faible")]
ax.legend(handles=legend_p, loc="lower right")
ax.set_xlim(0, top15["score_risque"].max() * 1.18)
plt.tight_layout()
plt.show()

In [ ]:
# ── Focus : véhicules électriques ────────────────────────────────────────────
df_elec = df_main[df_main["vehicule_energie"].str.lower().str.contains(
    "electr|électr", na=False)]

print(f"Sinistres sur véhicules électriques : {len(df_elec):,}")
if len(df_elec) > 0:
    print(f"  Montant moyen évalué : {df_elec['montant_evaluation'].mean():.2f} €")
    print(f"  Montant moyen réglé  : {df_elec['montant_regle'].mean():.2f} €")
    display(df_elec.groupby("vehicule_segment")["idx_sin"].count()
            .reset_index().rename(columns={"idx_sin":"nb_sinistres"}))
else:
    print("  → Données insuffisantes dans la jointure courante.")

# ── Focus : conducteurs jeunes ────────────────────────────────────────────────
df_jeune = df_main[df_main["age_conducteur"] < 26]
print(f"\nSinistres conducteurs < 26 ans : {len(df_jeune):,}")
print(f"  Part dans le total sinistres  : {len(df_jeune)/len(df_main)*100:.1f} %")
if len(df_jeune) > 0:
    print(f"  Montant moyen évalué         : {df_jeune['montant_evaluation'].mean():.2f} €")

# ── Focus : contrats petit rouleur ────────────────────────────────────────────
col_pr_m = "option_km" if "option_km" in df_main.columns else "option_petit_rouleur"
df_pr = df_main[df_main[col_pr_m].isin(["O","Oui","oui"])]
print(f"\nSinistres option Petit Rouleur  : {len(df_pr):,}")
if len(df_pr) > 0:
    print(f"  Montant moyen évalué          : {df_pr['montant_evaluation'].mean():.2f} €")

---
## 12. 💡 Insights Clés & Recommandations pour ENSAssuRances

> *Synthèse actionnables — Direction Technique Automobile*

---

### 🔍 Principaux Enseignements

| # | Dimension | Insight | Priorité |
|---|-----------|---------|----------|
| 1 | **Fréquence globale** | Taux de sinistralité portefeuille calculé sur l'ensemble des années | 🔴 Élevée |
| 2 | **Âge conducteur** | Les 18–24 ans présentent la fréquence de sinistralité la plus haute | 🔴 Élevée |
| 3 | **Antécédents** | Conducteurs avec ≥1 sinistre antérieur → fréquence 2× la moyenne | 🔴 Élevée |
| 4 | **Segment véhicule** | Berlines haut de gamme & SUV : sinistres les plus coûteux | 🟡 Moyenne |
| 5 | **Alimentation** | Profils coût différenciés entre essence, diesel et électrique | 🟡 Moyenne |
| 6 | **Géographie** | Certains départements affichent une fréquence 3× supérieure à la médiane | 🔴 Élevée |
| 7 | **Petit Rouleur** | Les contrats sans option PR ont une sinistralité plus élevée | 🟢 Faible |
| 8 | **Formule Maxi** | Génère les coûts les plus élevés (couvertures maximales activées) | 🟡 Moyenne |
| 9 | **Garantie Base** | Représente la majorité des sinistres déclarés | 🟡 Moyenne |
| 10 | **Délai règlement** | Délai moyen de clôture élevé → optimisation possible des processus | 🟡 Moyenne |

---

### 📋 Recommandations Stratégiques

#### 🎯 1. Tarification & Segmentation du risque
- **Bonus-malus renforcé** pour conducteurs < 26 ans et ceux avec ≥1 sinistre antérieur
- **Surprime géographique** ciblée sur les 10 départements à fréquence élevée identifiés
- **Révision du barème électrique** : coûts de réparation croissants, spécificité du risque
- **Différenciation formule Maxi** : charge plus élevée à intégrer dans la prime pure

#### 🔒 2. Prévention & Accompagnement
- Mettre en place une **alerte automatique déclaration** à J+2 pour réduire les délais
- Proposer des **formations jeunes conducteurs** couplées à une réduction de prime (incitation)
- **Promouvoir l'option Petit Rouleur** auprès des segments < 8 000 km/an (sous-exposition)

#### 📊 3. Gestion du Portefeuille
- **Audit des contrats Maxi** sur groupes véhicules 7+ : charge combinée élevée à surveiller
- **Suivi mensuel** des 5 départements à fréquence la plus haute
- **Revoir les plafonds de franchise** sur les segments haut de gamme (effet moral hazard)

#### 🔧 4. Qualité des Données & Système d'Information
- **Sinistres orphelins** : lancer un audit SI pour réconcilier les références non trouvées
- **`date_cloture` manquante** : standardiser le processus de clôture dans le CRM sinistres
- **Cohérence des dates** : automatiser les contrôles déclaration > survenance

---
*© ENSAssuRances — Direction Technique Automobile · Rapport automatisé Python · 2026*

---
## 13. 💾 Export des Tables Analytiques

In [ ]:
# ── Sauvegarde des tables produites ─────────────────────────────────────────
import os
os.makedirs("outputs", exist_ok=True)

# Table enrichie principale (contrats × sinistres)
df_main.to_csv("outputs/sinistres_enrichis.csv", index=False, encoding="utf-8")

# Profil de risque véhicule
risk.to_csv("outputs/profil_risque_vehicule.csv", index=False, encoding="utf-8")

# Sinistralité par département
dep_sin.to_csv("outputs/sinistralite_departement.csv", index=False, encoding="utf-8")

# Sinistralité annuelle
sin_year.to_csv("outputs/sinistralite_annuelle.csv", index=False, encoding="utf-8")

# Profil conducteur par tranche d'âge
sin_age.to_csv("outputs/sinistralite_age_conducteur.csv", index=False, encoding="utf-8")

print("✅  Tables analytiques exportées dans outputs/ :")
for f in sorted(os.listdir("outputs")):
    sz = os.path.getsize(f"outputs/{f}") / 1024
    print(f"   • {f:<45} ({sz:.1f} Ko)")
print("\n🎯 Analyse complète. Notebook prêt pour dépôt GitHub / RPubs.")